In [ ]:
#УСТАНОВКА ЗАВИСИМОСТЕЙ
!pip install dataparser deep-translator pandas  playwright asyncio nest_asyncio bs4 chromium
!playwright install --with-deps chromium

In [ ]:
base_link = "https://www.cian.ru/cat.php?deal_type=sale&engine_version=2&offer_type=flat&region=1&object_type%5B0%5D=2" # базовая ссылка объявлений по новостройкам cian

links = []

def generate_range(start, end, step):
    """Генерирует ссылки для диапазона цен"""
    curr = start
    while curr < end:
        #верхняя граница-1
        next_val = min(curr + step, end)
        max_p = next_val - 1

        url = f"{base_link}&minprice={curr}&maxprice={max_p}"
        links.append(url)

        curr = next_val
    print(len(links))

links.append('url')

#все до 6 млн берем вместе
generate_range(0, 6000000, 6000000)

#самый маленький шаг для минимума
generate_range(6_000_000, 25_000_000, 500_000)

#квартриры подороже - шаг 1 млн
generate_range(25_000_000, 50_000_000, 1_000_000)

#элита - шаг 5 млн
generate_range(50_000_000, 200_000_000, 5_000_000)

#элита покруче - шаг 50 млн
generate_range(200_000_000, 1_000_000_000, 50_000_000)


#остаток от 1 млрд
url = f"{base_link}&minprice=1000000000"
links.append(url)

#сохраняем полученные файлы
filename = "links_min_max.csv"
with open(filename, "w", encoding="utf-8") as f:
    for link in links:
        f.write(link + "\n")

print(f"Сгенерировано {len(links)} ссылок.")
print(f"Сохранено в файл: {filename}")

In [ ]:
from playwright.async_api import async_playwright
import asyncio
import nest_asyncio

#colab
nest_asyncio.apply()

async def fetch_html_simple(url):
    """Асинхронная функция для получения HTML"""

    print(f"Начинаем загрузку страницы: {url}")

    async with async_playwright() as p:
        # Запускаем браузер хромиум
        browser = await p.chromium.launch(
            headless=True,
            #args=['--no-sandbox', '--disable-dev-shm-usage', '--disable-gpu'] # если медленно работает, то включить этот args
            args=[
                '--no-sandbox',
                '--disable-dev-shm-usage',
                '--disable-gpu',
                '--disable-blink-features=AutomationControlled',
                '--disable-features=site-per-process',
                '--disable-extensions',
                '--disable-notifications',
                '--disable-sync',
                '--disable-background-networking',
                '--disable-default-apps',
                '--disable-translate',
                '--metrics-recording-only',
                '--no-first-run',
                '--mute-audio',
                '--hide-scrollbars',
            ]
        )

        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            java_script_enabled=True
        )

        page = await context.new_page()

        await page.goto(url, wait_until='domcontentloaded')

        await page.wait_for_timeout(200)

        html = await page.content()

        await browser.close()

        return html


In [ ]:
#Генерируем ссылки (с количеством страниц) для ссылок диапазона min-max. Так как cian выдаёт только 54 страницы лимитом на...
#... показ объявлений пользователю

import time
import pandas as pd
import os
from bs4 import BeautifulSoup
import re
import math

FILENAME = "links_min_max.csv"
OUTPUT_FILE = "checked_links.csv"

results = []

df = pd.read_csv(FILENAME)

# Создаем список для хранения всех ссылок
all_links = []

links = df['url'].dropna().tolist()
print(f"Всего ссылок в файле: {len(links)}")

for i, url in enumerate(links):
    print(f"Обработка ссылки {i+1}/{len(links)}: {url}")

    html = await fetch_html_simple(url)
    soup = BeautifulSoup(html, 'html.parser')
    header = soup.find('div', {'data-name': 'SummaryHeader'})

    if header:
        text = header.get_text()
        match = re.search(r'Найдено\s+([\d\s]+)', text) # регуляркой ищем "Найдено N объявлений"
        if match:
            clean_number = match.group(1).replace(' ', '').replace('\xa0', '')
            number_of_pages = math.ceil(int(clean_number)/28)
            links_with_page = [f"{url}&p={digit}" for digit in range(1, number_of_pages + 1)]

            # Добавляем все ссылки в общий список
            all_links.extend(links_with_page)

            print(f"Сгенерировано ссылок: {len(links_with_page)}")
            print(f"Всего накоплено ссылок: {len(all_links)}")

    else:
        print(f"Проблемная ссылка: {url} (не найден заголовок)")
        continue

#сейвим
df_to_save = pd.DataFrame({'url': all_links})

df_to_save.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"\n{'='*50}")
print(f"Всего сгенерировано ссылок: {len(all_links)}")
print(f"Результат сохранен в файл: {OUTPUT_FILE}")

In [ ]:
#Извлекаем со сгенерированных ссылок по количеству страниц на диапазон min-max ссылки на объявления о продаже недвижимости

from playwright.async_api import async_playwright
import asyncio
import nest_asyncio
import pandas as pd
from bs4 import BeautifulSoup
from typing import List
import re

#Для Colab
nest_asyncio.apply()

async def fetch_html_with_playwright(url, page_number):
    """
    Получает полностью загруженный HTML страницы через Playwright.
    Args:
        url: ссылка на объявление
        page_number: номер страницы
    """
    print(f"Загружаем страницу {page_number}: {url}")

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                '--no-sandbox',
                '--disable-dev-shm-usage',
                '--disable-gpu',
                '--disable-blink-features=AutomationControlled',
                '--disable-features=site-per-process',
                '--disable-extensions',
                '--disable-notifications',
                '--disable-sync',
                '--disable-background-networking',
                '--disable-default-apps',
                '--disable-translate',
                '--metrics-recording-only',
                '--no-first-run',
                '--mute-audio',
                '--hide-scrollbars',
            ] # настройки виртуального браузера для повышения быстродействия
        )

        context = await browser.new_context(
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            viewport={'width': 1920, 'height': 1080},
            java_script_enabled=True
        ) # user_agent чтобы не банилось IP


        page = await context.new_page()

        try:
            await page.goto(url, wait_until='commit') # бегаем по url и собираем ссылки на объявления

            await page.wait_for_timeout(1000)

            try:
                await page.wait_for_selector('[data-name="LinkArea"]', timeout=1500) # ждём пока браузер прогрузит все ссылки на странице
            except:
                print("Ждем дополнительное время для загрузки...")
                await page.wait_for_timeout(1000)

            html = await page.content()

            return html

        except Exception as e:
            print(f"Ошибка при загрузке {url}: {e}")
            return ""

        finally:
            await browser.close()


def extract_links_with_bs4(html, url):
    """
    Извлекает ссылки на объявления из HTML с помощью BeautifulSoup.
    Фильтрует аукционные объявления.

    Args:
        html : прогруженная html страница
        url : ссылка на объявление
    """
    if not html:
        return []

    soup = BeautifulSoup(html, 'html.parser')
    all_links = []
    auction_links = []
    regular_links = []

    print(f"Парсим HTML размером: {len(html)} символов")

    offer_containers = soup.find_all(class_=re.compile(r'x31de4314--_416c6--container'))
    print(f"Найдено офферов по частичному классу: {len(offer_containers)}")

    for offer in offer_containers:

        link_tag = offer.find('a', href=True)
        link = link_tag['href'] if link_tag else None

        if not link:
            continue

        # Нормализуем ссылку
        if link.startswith('/'):
            link = f"https://www.cian.ru{link}"
        elif not link.startswith('http'):
            link = f"https://{link}"

        all_links.append(link)

        is_auction = False

        auction_text = offer.find(string=re.compile(r'Электронные торги|Аукцион|торги', re.IGNORECASE)) # ищем все аукционные метки объявления
        if auction_text:
            is_auction = True
            print(f"Найдено аукционное объявление: {auction_text.text if hasattr(auction_text, 'text') else auction_text}")

        if is_auction:
            auction_links.append(link)
        else:
            regular_links.append(link)

    print(f"\nСтатистика по странице:")
    print(f"Всего найдено ссылок: {len(all_links)}")
    print(f"Обычных объявлений: {len(regular_links)}")
    print(f"Аукционных объявлений: {len(auction_links)}")

    #для того, чтобы оценивать адекватность работы скрипта (кликать по найденным ссылкам)
    if regular_links:
        print(f"\nПримеры обычных ссылок (первые 3):")
        for i, regular_link in enumerate(regular_links[:3]):
            print(f"{i+1}. {regular_link}")

    return regular_links


async def fetch_links_from_page(url, page_number):
    """
    Основная функция: получает HTML через Playwright и парсит ссылки через BeautifulSoup.

    Args:
        url: URL страницы
        page_number: Номер страницы
    """
    print(f"\n{'='*50}")
    print(f"Обработка страницы {page_number}: {url}")


    html = await fetch_html_with_playwright(url, page_number)

    if not html:
        print("Не удалось получить HTML")
        return []

    links = extract_links_with_bs4(html, url)

    return links


async def process_all_pages_to_dataframe(urls):
    """
    Обрабатывает все страницы и возвращает DataFrame со ссылками на все найденные объявления.

    Args:
        urls: список ссылок для обработки
    """
    all_data = []

    for i, url in enumerate(urls, 1):
        print(f"\nОбрабатываем страницу {i}/{len(urls)}")

        links = await fetch_links_from_page(url, i)

        for link in links:
            all_data.append({
                'link': link,
                'source_url': url
            })

        #сейвимся
        if (i + 1) % 5 == 0:

          print("Пора бы сохраниться")
          df = pd.DataFrame(all_data)
          df.to_csv('cian_links1.csv')

          df = df.drop_duplicates(subset=['link'])

        if i < len(urls):
            await asyncio.sleep(0.2)

    return df

if __name__ == "__main__":

  df_with_links = pd.read_csv(OUTPUT_FILE)
  set_links = df_with_links['url'].drop_duplicates().to_list()
  df = asyncio.run(process_all_pages_to_dataframe(set_links))

  if not df.empty:
    df.to_csv('cian_links1.csv', index=False, encoding='utf-8')
    print(f"\nСохранено в 'cian_links1.csv'")

    print("\nСтатистика по страницам:")
    page_stats = df.groupby('page_number').size().reset_index(name='count')
    print(page_stats)
else:
    print("DataFrame пустой")

In [ ]:
# проверяем, что все ссылки собраны и дропаем дупликаты, если таковы нашлись

import pandas as pd
cian_data_test = pd.read_csv('cian_links1.csv')
cian_data_test['link'].drop_duplicates().head(-1)

In [ ]:
# Начало основного блока. В этой ячейке код ищет по первым LINKS_TO_CHECK объявлениям всевозможные атрибуты с объявлений ....
# ... и добавляет в словарь найденный атрибут. Многие атрибуты представлены в классе OfferSummaryInfoItem, но далее мы дополнительно...
# ... будем расширять итоговое количество полей, которых нет в представленном классе

import pandas as pd
from bs4 import BeautifulSoup
import re

file_urls = 'cian_links1.csv'

unique_fields = {}

LINKS_TO_CHECK = 25

df_cian = pd.read_csv(file_urls)
print(df_cian.info())

urls_only = df_cian['link'].to_list()

for url in range(LINKS_TO_CHECK):

  current_page = urls_only[url]

  print(f"Началась обработка страницы {url + 1}/{LINKS_TO_CHECK}")
  html = await fetch_html_simple(current_page)
  soupp = BeautifulSoup(html,'html.parser')

  types_of_description = soupp.find_all('div', {'data-name': 'OfferSummaryInfoItem'})

  for types in types_of_description:
    text_types = types.find_all('p')
    field_name = text_types[0].get_text(strip=True)
    example_value = text_types[1].get_text(strip=True)
    if field_name not in unique_fields:
      unique_fields[field_name] = example_value
      print(f"    Новое поле: '{field_name}' (Пример: {example_value})")

print("\n" + "="*40)
print("ИТОГОВЫЙ СПИСОК ДОСТУПНЫХ ПОЛЕЙ:")
for name, example in unique_fields.items():
  print(f"- {name}: {example}")


In [ ]:
# Конец основного блока. Здесь извлекаются всевозможные поля и атрибуты с объявления и передаются в соответствующую колонку df

from deep_translator import GoogleTranslator
from typing import Set, Dict, List
print(f"НАЧИНАЕМ ФОРМИРОВАТЬ DF")
from datetime import datetime
from bs4 import BeautifulSoup
import dataparser
import os
import re
import time

#
def get_photos_from_script(soup):

  """ Функция получения ссылки на фотографии из объявления """

  scripts = soup.find_all('script')
  found_urls = []
  for script in scripts:
      if script.string and '"photos":[' in script.string:
          urls = re.findall(r'"fullUrl":"([^"]+)"', script.string)
          clean_urls = [u.replace("\\u002F", "/") for u in urls]
          found_urls.extend(clean_urls)

  return list(set(found_urls))

def get_coords_from_script(soup):

  """ Функция получения геолокации по координатам из объявления """

  scripts = soup.find_all('script')
  for script in scripts:
      if script.string:
          #ищем подстроку "coordinates":{"lat":55.123,"lng":37.123} - ищем все такие куски
          matches = re.findall(r'"coordinates":\{"lat":([\d\.]+),"lng":([\d\.]+)\}', script.string)
          for lat, lng in matches:
              #если нули, то ничего не возвращаем
              if float(lat) != 0 and float(lng) != 0:
                  return float(lat), float(lng)
  return None, None

def parse_date_string(date_str):

  """ Функция извлечения даты публикации из объявления """

  clean_str = str(date_str).replace("Обновлено", "").replace(":", "", 1).strip()

  try:
      dt = dateparser.parse(clean_str, languages=['ru'])

      if dt:
          return dt.strftime("%Y-%m-%d %H:%M:%S")
      else:
          return date_str

  except Exception:
      return date_str


def translate_words_deep(word: str, target_lang: str = 'en'):
    """
    Переводит слова с использованием deep-translator.
    """
    translator = GoogleTranslator(source='ru', target=target_lang)

    try:
        translation = translator.translate(word).lower().replace(' ', '_')

    except Exception as e:
        print(f"Ошибка при переводе слова '{word}': {e}")

    return translation

all_data = []
urls = cian_data_test['link'].drop_duplicates().to_list()
set1 = set(urls)

EXTENDED_COLUMNS= ['url',
 'price',
 'numbers_of_rooms',
 'floor',
 'address',
 'description',
 'metro_info',
 'url_photos',
 'lat',
 'lng',
 'seller',
 'date',
 'offer_date',
 'housing_type',
 'total_area',
 'living_area',
 'kitchen_area',
 'ceiling_height',
 'bathroom',
 'view_from_the_windows',
 'number_of_elevators',
 'house_type',
 'parking',
 'finishing',
 'balcony/loggia',
 'ramp',
 'minor_owners',
 'maternity_capital_upon_purchase',
 'about_the_entrance'] # список всех дополнительных полей, встречающихся в объявлениях

#справочник полей - перевод
available_columns = {'Тип жилья': 'housing_type',
                     'Общая площадь': 'total_area',
                     'Жилая площадь': 'living_area',
                     'Площадь кухни': 'kitchen_area',
                     'Высота потолков': 'ceiling_height',
                     'Санузел': 'bathroom',
                     'Вид из окон': 'view_from_the_windows',
                     'Количество лифтов': 'number_of_elevators',
                     'Тип дома': 'house_type',
                     'Парковка': 'parking',
                     'Отделка': 'finishing',
                     'Балкон/лоджия': 'balcony/loggia',
                     'Пандус': 'ramp',
                     'Несовершеннолетние собственники': 'minor_owners',
                     'Материнский капитал при\xa0покупке': 'maternity_capital_upon_purchase',
                     'О\xa0подъезде': 'about_the_entrance'}


if os.path.exists('cian_data.csv'):

  print(f"Файл существует, вычисляем количество оставшихся ссылок...")
  df_full_data = pd.read_csv('cian_data.csv',header=0)
  df_links = df_full_data['url'].drop_duplicates().to_list()
  set2 = set(df_links)
  ostat = list(set1 - set2)
  LINKS_TOTAL = len(ostat)
  print(f"Осталось строк: {LINKS_TOTAL}")

else:
  print(f"Датафрейма не существует, создаём и начинаем парсить")

  df_full_data = pd.DataFrame(columns=EXTENDED_COLUMNS)
  drive_path = '/content/drive/My Drive/'

  print(f"Создан пустой датафрейм с колонками")

  #сейвим
  df_full_data.to_csv(drive_path + 'cian_data.csv', index=False, mode='w',header=True)
  df_full_data.to_csv('cian_data.csv',mode='w',index=False,header=True)
  ostat = urls.copy()

  LINKS_TOTAL = len(urls)

#запускаем цикл извлечения атрибутов по оставшимся ссылкам
for url in range(LINKS_TOTAL):

    start_time = time.time()

    current_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    current_page = ostat[url]

    print(f"Началась обработка страницы {url + 1}/{LINKS_TOTAL}")

    page_data = {}

    html = await fetch_html_simple(current_page)
    soupp = BeautifulSoup(html, 'html.parser')

    page_data['url'] = current_page

    price = soupp.find_all('div', {'data-testid': 'price-amount'})
    if price:
        price_value = price[0].get_text(strip=True).replace('\xa0','').strip('₽')
        page_data['price'] = price_value

    #ишем все атрибуты класса OfferSummaryInfoItem в объявлении
    types_of_description = soupp.find_all('div', {'data-name': 'OfferSummaryInfoItem'})

    if types_of_description:
      for types in types_of_description:
          text_types = types.find_all('p')

          if len(text_types) >= 2:
            #лезем в переводчик только если поле отсутствует в словаре
            if text_types[0].get_text(strip=True) in available_columns.keys():
                field_name = available_columns[text_types[0].get_text(strip=True)]
            else:
                field_name = translate_words_deep(text_types[0].get_text(strip=True))
            example_value = text_types[1].get_text(strip=True)
            if field_name in df_full_data.columns:
                if example_value and field_name:
                  page_data[field_name] = example_value
                else:
                  continue
            else:
                continue

    rooms = soupp.find_all('div', {'data-name': 'OfferTitleNew'})
    count_room_text = ''

    #определяем количество комнат либо в цифре, либо типом
    for room in rooms:
        text_room = room.find_all('h1')
        if text_room:
            room_field = text_room[0].get_text(strip=True)
            pattern = r'\d+'
            matched = re.search(pattern, room_field)
            if matched:
                if int(matched.group()) >= 10:
                    pattern_studio = 'студ'
                    pattern_mngkv = 'много'
                    pattern_free = 'свобод'
                    if pattern_studio in room_field.lower():
                        count_room_text = 'Студия'
                    if pattern_free in room_field.lower():
                        count_room_text = 'Свободная'
                    elif pattern_mngkv in room_field.lower():
                        count_room_text = 'Многокомнатная'
                else:
                    count_room_text = matched.group()
                page_data['numbers_of_rooms'] = count_room_text

    #определяем этаж
    floor_info = soupp.find('div', {'data-name': 'ObjectFactoids'})
    if floor_info:
        floor_value = floor_info.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_16px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
        for floor_index in range(len(floor_value)):
            str_to_find = 'из'
            test = floor_value[floor_index].get_text(strip=True)
            match_floor = re.search(str_to_find, test)
            if match_floor:
                floor_text = test
                page_data['floor'] = floor_text
    #определяем адрес
    address = soupp.find_all('div', {'data-name': 'AddressContainer'})
    result = None
    for addr in address:
        text_addr = addr.find_all('a')
        if text_addr:
          result = ", ".join([g.text.strip() for g in text_addr])
          page_data['address'] = result
        else:
          page_data['address'] = result

    #забираем описание объявления
    desc_block = soupp.find('div', {'data-name': 'Description'})
    description = desc_block.get_text(separator=" ", strip=True) if desc_block else None
    page_data['description'] = description

    #извлекаем информацию по метро
    metro_block = soupp.find('ul', {'data-name': 'UndergroundList'})
    metro_info = metro_block.get_text(separator=", ", strip=True) if metro_block else None
    page_data['metro_info'] = metro_info

    #забираем ссылки на объявления через функцию
    raw_photos = get_photos_from_script(soupp)
    photos_str = ", ".join(raw_photos[:15])
    page_data['url_photos'] = photos_str

    #забираем координаты недвижимости
    lat, lng = get_coords_from_script(soupp)
    page_data['lat'] = lat
    page_data['lng'] = lng

    str_agent_info = ""

    #извлекаем информацию о продавце (ЖК, риэлторское агентсво, застройщик, собственник)
    jk_info =soupp.find('div', {'data-name': 'NewbuildingPremiumBuilderLogo'})
    estate_agency_info = soupp.find('div', {'data-name': 'AgencyBrandingAsideCardComponent'})
    estate_agency_layout = soupp.find('div', {'data-name': 'AgentInfoLayout'})
    developer_info = soupp.find('div', {'data-name' : 'DeveloperLogo'})
    owner_info = soupp.find('div', {'data-name' : 'HomeownerLayout'})#

    if jk_info:
      titles = jk_info.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_4u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_10px xa15a2ab7--_17731--display_inline-block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_textTransform__uppercase')
      values = jk_info.find_all('h3', class_='xa15a2ab7--_7735e--color_text-main-default xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_18px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
      if titles and values:
        text_titles = titles[0].get_text(strip=True)
        text_values = values[0].get_text(strip=True)
        str_agent_info = text_titles + ', ' + text_values
        print(f"Найден застройщик: {text_titles} наименование: {text_values}")

    if estate_agency_info:
      titles = estate_agency_info.find_all('span', class_='xa15a2ab7--_10cdd--color_gray60_100 xa15a2ab7--_2697e--lineHeight_4u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_10px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_textTransform__uppercase')
      values = estate_agency_info.find_all('span', class_='xa15a2ab7--_10cdd--color_current_color xa15a2ab7--_7735e--color_current_color xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_18px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
      if titles and values:
        text_titles = titles[0].get_text(strip=True)
        text_values = values[0].get_text(strip=True)
        str_agent_info = text_titles + ', ' + text_values
        print(f"Найдено агентсво: {text_titles} наименование: {text_values}")

    if estate_agency_layout:
      titles = estate_agency_layout.find_all('span', class_='xa15a2ab7--_10cdd--color_gray60_100 xa15a2ab7--_2697e--lineHeight_4u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_10px xa15a2ab7--_17731--display_inline-block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_textTransform__uppercase')
      values = estate_agency_layout.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_18px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
      if titles and values:
        text_titles = titles[0].get_text(strip=True)
        text_values = values[0].get_text(strip=True)
        str_agent_info = text_titles + ', ' + text_values
        print(f"Найден layout: {text_titles} наименование: {text_values}")

    if developer_info:
      titles = developer_info.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_4u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_10px xa15a2ab7--_17731--display_inline-block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_textTransform__uppercase')
      values = developer_info.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_18px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
      if titles and values:
        text_titles = titles[0].get_text(strip=True)
        text_values = values[0].get_text(strip=True)
        str_agent_info = text_titles + ', ' + text_values
        print(f"Найден developer_info: {text_titles} наименование: {text_values}")

    if owner_info:
      titles = owner_info.find_all('span', class_='xa15a2ab7--_10cdd--color_gray60_100 xa15a2ab7--_2697e--lineHeight_4u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_10px xa15a2ab7--_17731--display_inline-block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_textTransform__uppercase')
      values = owner_info.find_all('span', class_='xa15a2ab7--_7735e--color_text-primary-default xa15a2ab7--_2697e--lineHeight_6u xa15a2ab7--_2697e--fontWeight_bold xa15a2ab7--_2697e--fontSize_18px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text')
      if titles and values:
        text_titles = titles[0].get_text(strip=True)
        text_values = values[0].get_text(strip=True)
        str_agent_info = text_titles + ', ' + text_values
        print(f"Найден owner: {text_titles} наименование: {text_values}")

    #извлекаем дату публикации
    publication_data = soupp.find('div', {'data-testid': 'metadata-updated-date'})# OfferMetaData
    if publication_data:
      publication_data_text_to_search = publication_data.find_all('span',class_='xa15a2ab7--_10cdd--color_gray40_100 xa15a2ab7--_2697e--lineHeight_5u xa15a2ab7--_2697e--fontWeight_normal xa15a2ab7--_2697e--fontSize_14px xa15a2ab7--_17731--display_block xa15a2ab7--dc75cc--text xa15a2ab7--dc75cc--text_letterSpacing__0')
      publication_data_text = publication_data_text_to_search[0].get_text(strip=True)
      print(f"Publication date: {publication_data_text}")

    page_data['seller'] = str_agent_info
    page_data['date'] = current_date
    page_data['offer_date'] = parse_date_string(publication_data_text) if publication_data else None

    end_time = time.time()
    execution_time = end_time - start_time # определяем время, затрачиваемое на парсинг текущей страницы

    print(f'Время обработки ссылки - {execution_time:.2f} сек.')

    all_data.append(page_data)

    if (url + 1) % 5 == 0:
      print("Пора бы сохраниться")

      df_full_data = pd.DataFrame(all_data, columns=EXTENDED_COLUMNS)
      drive_path = '/content/drive/My Drive/'
      df_full_data.to_csv(drive_path + 'cian_data.csv', index=False, mode='a',header=False)
      df_full_data.to_csv('cian_data.csv',mode='a',index=False,header=False)
      all_data = []

      # Цена в объявлениях присутсвует всегда, её не может не быть. Если парсер не нашёл цену, значит произошла...
      # ... какая-либо ошибка (н-р бан IP). Поэтому, если такая ошибка встречается, то парсер её показывает и завершает принудительно работу
      print("Проводим тест адекватности работы скрипта...")
      price_test = df_full_data['price']
      if price_test.isna().any():
        print("Что-то пошло не так, прекращаем работу...")
        break
      else:
        print("Тест выполнен, всё хорошо")

df_full_data = pd.DataFrame(all_data, columns=EXTENDED_COLUMNS)
df_full_data.to_csv('cian_data.csv',mode='a',index=False,header=False)
df_full_data.to_csv(drive_path + 'cian_data.csv', index=False, mode='a',header=False)

print(f"Parsing is over")
print(df_full_data.info())
print(df_full_data.head())
print(f"Всего обработано страниц: {len(df_full_data)}")

In [ ]:
# Почти всё готово! Теперь необходимо распарсить полученные данные (н-р: этаж в объявлении указывается как "7 из 29"...
# ... количество этажей в доме 29 и тому подобное)

import pandas as pd
import re
import numpy as np
from datetime import datetime, timedelta
import re

cian_df = pd.read_csv('cian_data.csv')
print(f"Оригинальная размерность датафрейма: {cian_df.shape}")
cian_cleaned = cian_df.dropna(subset=['price'])

def parse_data_with_patterns(text,pattern1,pattern2, numbers : bool):

  """ Функция разбивает данные по указанным паттернам """

  if pd.isna(text):
      return pd.Series([np.nan, np.nan])

  text = str(text).strip().lower()

  sovm = np.nan
  razdel = np.nan

  sovm_match = re.search(pattern1, text)
  razdel_match = re.search(pattern2, text)

  if numbers:
    if sovm_match:
        sovm = int(sovm_match.group(1))

    if razdel_match:
        razdel = int(razdel_match.group(1))

    return pd.Series([sovm, razdel])

  else:
    if sovm_match:
      sovm = 'Yes'
    if razdel_match:
      razdel = 'Yes'
    return pd.Series([sovm, razdel])

def parse_housing_type(text,pattern1,pattern2):

  """Функция для определения типа дома"""

  orig_text = str(text)

  if pd.isna(text):
        return pd.Series([np.nan, np.nan])

  text = str(text).strip().lower()

  new_building_text = np.nan
  appartment_text = np.nan

  new_building_text = re.search(pattern1, text)
  appartment_text = re.search(pattern2, text)

  if new_building_text:

    return pd.Series([orig_text, appartment_text])

  if appartment_text:

    return pd.Series([new_building_text, orig_text])

  return pd.Series([new_building_text, appartment_text])


def get_publication_date(row):
  """ Определяет дату публикации на основе данных из двух колонок """
  scraped_date = row[0]
  update_info = str(row[1])
  scraped_date = datetime.strptime(scraped_date, '%Y-%m-%d %H:%M:%S')

  # определяем точную дату публикации объявления
  # Если обновлено:
  if update_info.startswith('Обновлено: '):
    update_info = update_info.replace('Обновлено: ', '')

  # Если (к примеру) "вчера, 21:40"
  if 'вчера' in update_info.lower():
    time_match = re.search(r'(\d{1,2}):(\d{2})', update_info)
    if time_match:
        hour, minute = map(int, time_match.groups())
        pub_date = scraped_date - timedelta(days=1)
        pub_date = pub_date.replace(hour=hour, minute=minute, second=0)
        return pub_date

  # Если (к примеру) "сегодня, 06:55"
  elif 'сегодня' in update_info.lower():
      # Извлекаем время из второй колонки
      time_match = re.search(r'(\d{1,2}):(\d{2})', update_info)
      if time_match:
          hour, minute = map(int, time_match.groups())
          # Берем дату сбора, меняем только время
          pub_date = scraped_date.replace(hour=hour, minute=minute, second=0)
          return pub_date

  else:

      match = re.search(r'(\d{1,2})\s+([а-я]{3})[,\s]+(\d{1,2}):(\d{2})', update_info, re.IGNORECASE)
      if match:
          day, month_str, hour, minute = match.groups()
          day = int(day)
          hour = int(hour)
          minute = int(minute)

          # Словарь для преобразования месяцев
          months = {
                'янв': 1, 'фев': 2, 'мар': 3, 'апр': 4,
                'май': 5, 'июн': 6, 'июл': 7, 'авг': 8,
                'сен': 9, 'окт': 10, 'ноя': 11, 'дек': 12
          }

          month = months.get(month_str.lower())
          if month:

              year = scraped_date.year

              try:
                  pub_date = datetime(year, month, day, hour, minute, 0)
                  return pub_date

              except:
                  pass

    # Если ничего не подошло, возвращаем дату сбора
  return scraped_date

print(f"Размерность датафрейма после удаления пустых страниц: {cian_cleaned.shape}")

#распарсинг данных, удаление старых и формирование новых столбцов

cian_cleaned[['floor_number', 'total_floors']] = cian_cleaned['floor'].str.split(' из ', expand=True)
del cian_cleaned['floor']

cian_cleaned['floor_number'] = cian_cleaned['floor_number'].astype(int)
cian_cleaned['total_floors'] = cian_cleaned['total_floors'].astype(int)

cian_cleaned['total_area'] = cian_cleaned['total_area'].str.replace('\xa0м²','') \
                             .str.replace(',','.') \
                             .astype(float)
cian_cleaned['living_area'] = cian_cleaned['living_area'].str.replace('\xa0м²','') \
                             .str.replace(',','.') \
                             .astype(float)
cian_cleaned['kitchen_area'] = cian_cleaned['kitchen_area'].str.replace('\xa0м²','') \
                             .str.replace(',','.') \
                             .astype(float)
cian_cleaned['ceiling_height'] = cian_cleaned['ceiling_height'].str.replace('\xa0м','') \
                             .str.replace(',','.') \
                             .astype(float)

comb_bathroom_pattern = r'(\d+)\s*[-–]?\s*совмещен'
razdel_pattern = r'(\d+)\s*[-–]?\s*раздел'

cian_cleaned[['combined_bathroom', 'separated_bathroom']] = cian_cleaned['bathroom'].apply(lambda x: parse_data_with_patterns(x,comb_bathroom_pattern,razdel_pattern,True))
del cian_cleaned['bathroom']

passanger_pattern = r'(\d+)\s*[-–]?\s*пассажир'
cargo_pattern = r'(\d+)\s*[-–]?\s*грузов'
cian_cleaned[['passenger_elevator', 'cargo_elevator']] = cian_cleaned['number_of_elevators'].apply(lambda x: parse_data_with_patterns(x,passanger_pattern,cargo_pattern,True))
del cian_cleaned['number_of_elevators']

balcony_pattern = r'(\d+)\s*[-–]?\s*балко'
loggia_pattern = r'(\d+)\s*[-–]?\s*лодж'
cian_cleaned[['balcony', 'loggia']] = cian_cleaned['balcony/loggia'].apply(lambda x: parse_data_with_patterns(x,balcony_pattern,loggia_pattern,True))
del cian_cleaned['balcony/loggia']

garbage_pattern = 'мусор'
concierge_pattern = 'консь'
cian_cleaned[['garbage_chute', 'concierge']] = cian_cleaned['about_the_entrance'].apply(lambda x: parse_data_with_patterns(x,garbage_pattern,concierge_pattern,False))
del cian_cleaned['about_the_entrance']

new_building_pattern = 'новостр'
apartment_pattern = 'вторич'

cian_cleaned[['new_building', 'apartment']] = cian_cleaned['housing_type'].apply(lambda x: parse_housing_type(x,new_building_pattern,apartment_pattern))
del cian_cleaned['housing_type']

cian_cleaned['date'] = pd.to_datetime(cian_cleaned['date'], format='%Y-%m-%d %H:%M:%S')
cian_cleaned['publication_date'] = cian_cleaned[['date','offer_date']].apply(get_publication_date,axis=1)

#сейвим итоговый результат
cian_cleaned.to_csv('cian_data_filtered.csv',mode='w',index=False,header=True)